# TARP and skill tables

Reads the HDF5 files written by `CFM_TARP_RMSE_Eval.py` and writes the
appendix tables of the paper as LaTeX, one `.tex` file per table:

| kind | one table per | columns |
| --- | --- | --- |
| `tarp`  | dataset x measurement class | $\max|\Delta|$, $c$, $\sigma_{\rm TARP}$, $|\Delta|/\sigma_{\rm TARP}$ |
| `skill` | dataset x measurement class | $1 - \mathrm{RMSE}/\mathrm{RMSE_{baseline}}$ per parameter |

One row per conditioning pattern, in the eval script's pattern order. A
pattern appears in a class's tables only when its TARP curve for that class
was computed (status `ok`).

**RMSE comes from the class pool** (`RMSE_SOURCE = 'class'`, the default):
the same sources and the same posterior draws as that class's TARP curve,
scored on the TARP truth vector (class features minus conditioning
features). `RMSE_SOURCE = 'pattern'` falls back to the older pattern-pool
RMSE for runs made before `rmse_class_table/` existed.

**TARP numbers.** For each curve, $\Delta(c) = \mathrm{ECP}(c) - c$ on the
script's credibility grid $c = k/(N+1)$. The table reports the signed
$\Delta$ at the largest $|\Delta|$, the $c$ where it occurs,
$\sigma_{\rm TARP} = \sqrt{c(1-c)/n}$ there, and the ratio.

The files use `booktabs` and `longtable`, matching the paper's preamble.
`\input{}` each one where it belongs.

In [1]:
# =====================================================================
# CONFIGURATION -- edit this cell
# =====================================================================
from pathlib import Path

# Folder holding the eval outputs (the script's --out-dir):
#   <RUN_TAG>_TARP_RMSE_<dataset>_<timestamp>.h5
RESULT_DIR = Path('INSERT')
TIMESTAMP = '2026-09-24_12-01'        # pin one run, e.g. '2026-09-24_12-01'; None -> newest
RUN_TAG = 'RMSE_ON_CLASSES'          # e.g. 'RMSE_ON_CLASSES'; None -> any run tag

# Datasets to tabulate. 
DATASETS = ['test', 'omega_cen_not_A']     # any of 'test', 'omega_cen_all', 'omega_cen_not_A'

# Where the .tex files go: <kind>_<dataset>_<class letter>.tex,
# e.g. tarp_test_a.tex, skill_test_d.tex.
TABLE_DIR = Path('tables_tarp_skill')

# 'class'   : RMSE on the class pool, the same sources and draws as the TARP
#             curve (needs a run that wrote rmse_class_table/).
# 'pattern' : the older pattern-pool RMSE (general pool, RV pool for v_r),
#             masked to the class truth vector.  For old runs only.
RMSE_SOURCE = 'class'

KINDS = ['tarp', 'skill']       # which table kinds to write
CLASSES_TO_WRITE = None         # None -> all four; or e.g. ['c_pos_phot_astro']

# ---- LaTeX cross-references ------------------------------------------
LABEL_FMT = 'tab:{kind}:{dataset}:{letter}'   # -> tab:tarp:test:a
EQ_DEV_LABEL = 'eqn:max_dev_tarp'             # |Delta| = |ECP - c|
EQ_ERR_LABEL = 'eqn:TARP_error'               # sigma_TARP

# ---- Number formatting -------------------------------------------------
DEV_DECIMALS = 3        # max|Delta|, signed
CRED_DECIMALS = 2       # c
SIGMA_DECIMALS = 3      # sigma_TARP
RATIO_DECIMALS = 1      # |Delta| / sigma_TARP
SKILL_DECIMALS = 2      # skill, signed
NA = '--'               # empty cell (renders as an en dash)
TABCOLSEP = '4pt'       # column padding; None leaves the document default
SHOW_N = 'auto'         # 'auto': add an n column only when n differs between
                        # rows of a table; True / False force it on / off

# ---- Extra outputs -----------------------------------------------------
WRITE_PREVIEW = True    # TABLE_DIR/tables_preview.tex, a standalone document
                        # that \input's every table, for checking the layout
SAVE_CSV = True         # the unrounded numbers behind the tables

# ---- Caption wording ---------------------------------------------------
# How each dataset is named inside a caption.
DS_CAPTION = {'test': 'the test set',
              'omega_cen_all': r'$\omega$~Cen members',
              'omega_cen_not_A': r'$\omega$~Cen members outside training/validation set A'}

# Caption templates.  Tokens in <<...>> are filled in per table:
#   <<N>>            number of sources ('up to 1000' when n varies)
#   <<DATASET>>      DS_CAPTION[dataset]
#   <<CLASS>>        class letter, e.g. c
#   <<FEATURES>>     the class features as symbols
#   <<FIRST_TABLE>>  label of the first table of the same kind and dataset,
#                    which carries the full description
#   <<EQ_DEV>>, <<EQ_ERR>>   the equation labels above
#   <<RMSE_POOL>>    RMSE_POOL_SENTENCE[RMSE_SOURCE]
# 'full' goes on the first table of each kind and dataset, 'short' on the
# rest, 'cont' on the pages a long table continues onto.
TARP_CAPTION = {
    'full': (r'TARP deviation for <<N>> sources from <<DATASET>>. Calculated on '
             r'class (<<CLASS>>), i.e., sources with measured <<FEATURES>> only. '
             r'Cond.\ Pattern denotes the specific set of conditioning '
             r'information provided to the model when sampling. $\max|\Delta|$ '
             r'is the maximum deviation, where the deviation is defined in '
             r'Equation~\ref{<<EQ_DEV>>}; its sign is that of ECP~$-$~$c$ at '
             r'the maximum. $c$ is the credibility corresponding to the maximum '
             r'deviation. $\sigma_{\rm TARP}$ is the error on the TARP '
             r'measurement at $c$, defined in Equation~\ref{<<EQ_ERR>>}. '
             r'$|\Delta|/\sigma_{\rm TARP}$ is the maximum deviation in units of '
             r'that error.'),
    'short': (r'TARP deviation for <<N>> sources from <<DATASET>>. Calculated on '
              r'class (<<CLASS>>), i.e., sources with measured <<FEATURES>> only. '
              r'See the caption of Table~\ref{<<FIRST_TABLE>>} for a full '
              r'description of the columns.'),
    'cont': (r'TARP deviation for <<DATASET>>, class (<<CLASS>>), <<N>> sources '
             r'with measured <<FEATURES>> only (continued)'),
}
SKILL_CAPTION = {
    'full': (r'Skill ($1 -$ RMSE / RMSE$_{\rm baseline}$) for <<DATASET>>, class '
             r'(<<CLASS>>), <<N>> sources with measured <<FEATURES>> only. Each '
             r'column beyond Cond.\ Pattern corresponds to a single parameter, '
             r'and each row to a conditioning pattern. <<RMSE_POOL>> When a '
             r'parameter is included in the conditioning pattern, no RMSE is '
             r"calculated and its entry is marked with ``--''."),
    'short': (r'Skill ($1 -$ RMSE / RMSE$_{\rm baseline}$) for <<DATASET>>, class '
              r'(<<CLASS>>), <<N>> sources with measured <<FEATURES>> only. See '
              r'the caption of Table~\ref{<<FIRST_TABLE>>} for a full '
              r'description.'),
    'cont': (r'Skill ($1 -$ RMSE / RMSE$_{\rm baseline}$) for <<DATASET>>, class '
             r'(<<CLASS>>), <<N>> sources with measured <<FEATURES>> only '
             r'(continued)'),
}
RMSE_POOL_SENTENCE = {
    'class': (r'The RMSE and the baseline are computed on the same sources and '
              r'model samples as the TARP curve for this class.'),
    'pattern': (r'The RMSE and the baseline are computed on all sources with the '
                r'conditioning parameters measured, not only those of this class.'),
}

In [2]:
# =====================================================================
# Imports, symbols and small helpers
# =====================================================================
import json
import re

import h5py
import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_rows', 200)

# Symbol bodies without $ delimiters, so a list of features joins into ONE
# math expression: '$\ell,\,b,\,\varpi$'.
SYM = {'l': r'\ell', 'b': r'b', 'parallax': r'\varpi',
       'pml': r'\mu_{\ell*}', 'pmb': r'\mu_b', 'phot_g_mean_mag': r'G',
       'phot_bp_mean_mag': r'G_{BP}', 'phot_rp_mean_mag': r'G_{RP}',
       'radial_velocity': r'v_r'}
# Plain-text names for the on-screen previews.
UNI = {'l': 'l', 'b': 'b', 'parallax': 'plx', 'pml': 'pml', 'pmb': 'pmb',
       'phot_g_mean_mag': 'G', 'phot_bp_mean_mag': 'BP',
       'phot_rp_mean_mag': 'RP', 'radial_velocity': 'RV'}


def math(features):
    """Features as one math expression, e.g. '$\\ell,\\,b,\\,\\varpi$'."""
    if isinstance(features, str):
        features = [features]
    return '$' + r',\,'.join(SYM.get(f, f) for f in features) + '$'


def short(f):
    return UNI.get(f, f)


def short_pattern(cond, all_features, symbols=True):
    """'uncond', 'drop $\\varpi$', 'only $G$', or the conditioning set."""
    name = math if symbols else short
    if not cond:
        return 'uncond'
    if len(cond) == len(all_features) - 1:
        missing = [f for f in all_features if f not in cond][0]
        return 'drop ' + name(missing)
    if len(cond) == 1:
        return 'only ' + name(cond[0])
    return math(cond) if symbols else ','.join(short(f) for f in cond)


def class_letter(cls):
    """'c_pos_phot_astro' -> 'c'."""
    return cls.split('_', 1)[0]


def _attr(obj, key, default=None):
    """An HDF5 attribute, decoded from bytes and from JSON where it is JSON."""
    if key not in obj.attrs:
        return default
    v = obj.attrs[key]
    if isinstance(v, bytes):
        v = v.decode()
    if isinstance(v, str):
        try:
            return json.loads(v)
        except (ValueError, TypeError):
            return v
    return v


def fill(template, **tokens):
    """Replace <<TOKEN>> placeholders; LaTeX braces are left alone."""
    out = template
    for k, v in tokens.items():
        out = out.replace(f'<<{k}>>', str(v))
    left = re.findall(r'<<[A-Z_]+>>', out)
    if left:
        raise KeyError(f'caption template has unfilled tokens {left}')
    return out

In [3]:
# =====================================================================
# Find one HDF5 file per dataset
# =====================================================================
RESULT_DIR = Path(RESULT_DIR).expanduser()
if not RESULT_DIR.is_dir():
    raise FileNotFoundError(f'RESULT_DIR does not exist: {RESULT_DIR}')

found = {}                                  # dataset -> (timestamp, path)
pattern = f'{RUN_TAG}_TARP_RMSE_*.h5' if RUN_TAG else '*_TARP_RMSE_*.h5'
for p in sorted(RESULT_DIR.glob(pattern)):
    stem = p.name[:-3].split('_TARP_RMSE_', 1)[1]        # <dataset>_<timestamp>
    for ds in DATASETS:
        if stem.startswith(ds + '_'):
            ts = stem[len(ds) + 1:]
            if TIMESTAMP in (None, ts) and (ds not in found or ts > found[ds][0]):
                found[ds] = (ts, p)

missing = [ds for ds in DATASETS if ds not in found]
if missing:
    raise FileNotFoundError(
        f'no {pattern} for {missing} in {RESULT_DIR}'
        + (f' at TIMESTAMP={TIMESTAMP}' if TIMESTAMP else ''))
for ds in DATASETS:
    print(f'  {ds:<16s} {found[ds][1].name}')

  omega_cen_not_A  RMSE_ON_CLASSES_TARP_RMSE_omega_cen_not_A_2026-09-24_12-01.h5


In [4]:
# =====================================================================
# Load TARP curves and RMSE, and reduce them to table numbers
# =====================================================================
def tarp_numbers(ecp, cred, n):
    """Signed deviation at the largest |ECP - c|, where it is, and its error.

    sigma = sqrt(c (1 - c) / n) is the standard deviation of ECP at that ONE
    credibility level for a calibrated model (ECP = K/n, K ~ Binomial(n, c)).
    It is 0 only at c = 0 or 1, where the ratio is undefined (NaN).
    """
    dev = np.asarray(ecp, float) - np.asarray(cred, float)
    k = int(np.argmax(np.abs(dev)))
    c = float(cred[k])
    sigma = float(np.sqrt(max(c * (1.0 - c), 0.0) / n))
    ratio = abs(dev[k]) / sigma if sigma > 0 else np.nan
    return dict(max_dev=float(dev[k]), cred_at_max=c, sigma=sigma, ratio=ratio)


TARP_ROWS, SKILL_ROWS, META = [], [], {}
for ds in DATASETS:
    ts, path = found[ds]
    with h5py.File(path, 'r') as h5:
        meta = dict(path=path, timestamp=ts, status=_attr(h5, 'status'),
                    features=list(_attr(h5, 'feature_names')),
                    classes=dict(_attr(h5, 'classes')),
                    patterns=list(_attr(h5, 'pattern_names')),
                    n_draws=int(_attr(h5, 'n_draws', 0)))
        META[ds] = meta
        if meta['status'] != 'complete':
            print(f'WARNING: {ds} has status={meta["status"]!r}; the run may '
                  'have been interrupted.')
        feats = meta['features']
        J = {f: j for j, f in enumerate(feats)}

        if RMSE_SOURCE == 'class':
            if 'rmse_class_table' not in h5:
                raise KeyError(
                    f'{path.name} has no rmse_class_table/: it was written by an '
                    "older CFM_TARP_RMSE_Eval.py.  Re-run the eval, or set "
                    "RMSE_SOURCE = 'pattern'.")
            rm_tables = {c: h5[f'rmse_class_table/{c}'] for c in meta['classes']}
        elif RMSE_SOURCE == 'pattern':
            rm_tables = {c: h5['rmse_table'] for c in meta['classes']}
        else:
            raise ValueError(f"RMSE_SOURCE must be 'class' or 'pattern', "
                             f'not {RMSE_SOURCE!r}')

        for i, pname in enumerate(meta['patterns']):
            grp = h5[f'patterns/{pname}']
            cond = list(_attr(grp, 'cond_features', []))
            for cls, cfeats in meta['classes'].items():
                tg = grp[f'tarp/{cls}']
                status = _attr(tg, 'status')
                if status != 'ok':
                    continue        # not applicable, empty, or too few sources
                n = int(_attr(tg, 'n_sources'))
                truth = [f for f in cfeats if f not in cond]
                base = dict(dataset=ds, cls=cls, pattern=pname, p_idx=i,
                            cond=cond, n=n)
                TARP_ROWS.append({**base, **tarp_numbers(tg['ecp'][()],
                                                         tg['credibility'][()], n)})

                t = rm_tables[cls]
                for f in cfeats:
                    j = J[f]
                    if f in truth:
                        rmse = float(t['rmse_phys_model'][i, j])
                        bl = float(t['rmse_phys_baseline'][i, j])
                        n_f = int(t['n_phys'][i, j])
                    else:
                        rmse = bl = np.nan          # conditioned on
                        n_f = 0
                    skill = (1.0 - rmse / bl) if (np.isfinite(rmse) and
                                                  np.isfinite(bl) and bl > 0) else np.nan
                    SKILL_ROWS.append({**base, 'feature': f, 'rmse': rmse,
                                       'baseline': bl, 'skill': skill, 'n_rmse': n_f})

TARP_DF = pd.DataFrame(TARP_ROWS)
SKILL_DF = pd.DataFrame(SKILL_ROWS)

FEATURES = META[DATASETS[0]]['features']
CLASSES = META[DATASETS[0]]['classes']            # name -> class features
print(f'RMSE source: {RMSE_SOURCE} pool')
for ds in DATASETS:
    m = META[ds]
    counts = TARP_DF.loc[TARP_DF['dataset'] == ds].groupby('cls').size()
    print(f"  {ds:<16s} {m['n_draws']} draws/source;  rows per class: "
          + ', '.join(f'({class_letter(c)}) {int(counts.get(c, 0))}' for c in CLASSES))

# In the class pool every scored feature is measured for every source, so
# its n must equal the TARP n.  A mismatch means the file is not what this
# notebook expects.
if RMSE_SOURCE == 'class' and not SKILL_DF.empty:
    scored = SKILL_DF.loc[SKILL_DF['rmse'].notna()]
    bad = scored.loc[scored['n_rmse'] != scored['n']]
    if len(bad):
        print(f'WARNING: {len(bad)} class-pool RMSE values were computed on a '
              'different n than their TARP curve')

RMSE source: class pool
  omega_cen_not_A  200 draws/source;  rows per class: (a) 0, (b) 0, (c) 29, (d) 47


## Preview

The numbers exactly as they will be printed, in plain text, before any
LaTeX is written. `preview('c_pos_phot_astro')` shows a single class.

In [5]:
def _fmt_signed(v, dec, tex=True):
    """'+0.029' / '-0.044' with a true minus in LaTeX; '0.000' never signed."""
    if v is None or not np.isfinite(v):
        return NA if tex else ''
    s = f'{abs(v):.{dec}f}'
    if float(s) == 0.0:
        return s
    sign = '+' if v > 0 else '-'
    return (f'${sign}$' + s) if tex else sign + s


def _fmt(v, dec, tex=True):
    if v is None or not np.isfinite(v):
        return NA if tex else ''
    return f'{v:.{dec}f}'


def table_frames(ds, cls):
    """The TARP rows and the wide skill rows for one table pair, in order."""
    t = (TARP_DF.loc[(TARP_DF['dataset'] == ds) & (TARP_DF['cls'] == cls)]
         .sort_values('p_idx'))
    s = SKILL_DF.loc[(SKILL_DF['dataset'] == ds) & (SKILL_DF['cls'] == cls)]
    wide = (s.pivot(index='p_idx', columns='feature', values='skill')
            .reindex(columns=CLASSES[cls]).reindex(t['p_idx']))
    return t, wide


def preview(cls=None, ds=None):
    for d in ([ds] if ds else DATASETS):
        for c in ([cls] if cls else CLASSES):
            t, wide = table_frames(d, c)
            if t.empty:
                print(f'\n{d}, class ({class_letter(c)}): no usable curves')
                continue
            lab = [short_pattern(x, FEATURES, symbols=False) for x in t['cond']]
            view = pd.DataFrame({
                'pattern': lab,
                'n': t['n'].to_numpy(),
                'max_dev': [_fmt_signed(v, DEV_DECIMALS, False) for v in t['max_dev']],
                'c': [_fmt(v, CRED_DECIMALS, False) for v in t['cred_at_max']],
                'sigma': [_fmt(v, SIGMA_DECIMALS, False) for v in t['sigma']],
                'ratio': [_fmt(v, RATIO_DECIMALS, False) for v in t['ratio']],
            })
            for f in CLASSES[c]:
                view[short(f)] = [_fmt_signed(v, SKILL_DECIMALS, False)
                                  for v in wide[f].to_numpy()]
            bar = '=' * 100
            print(f'\n{bar}\n{d}, class ({class_letter(c)}): measured '
                  f"{','.join(short(f) for f in CLASSES[c])}   "
                  f'{len(view)} patterns\n{bar}')
            print(view.to_string(index=False))


preview()


omega_cen_not_A, class (a): no usable curves

omega_cen_not_A, class (b): no usable curves

omega_cen_not_A, class (c): measured l,b,plx,pml,pmb,G,BP,RP   29 patterns
            pattern    n max_dev    c sigma ratio       l       b   plx   pml    pmb     G    BP    RP
             uncond 1000  -0.077 0.59 0.016   5.0 -227.08  -84.50 -0.14 -0.41  -9.98 -0.25 -0.52 -0.11
                l,b 1000  -0.119 0.67 0.015   8.0                 -0.24 -4.95  -9.55 -0.22 -0.41 -0.15
           only plx 1000  -0.091 0.71 0.014   6.4 -212.36  -88.17       -0.35 -10.28 -0.13 -0.38 -0.03
            l,b,plx 1000  -0.180 0.71 0.014  12.6                       -4.07  -9.94 -0.17 -0.29 -0.12
              l,b,G 1000  -0.216 0.82 0.012  17.8                 -0.29 -5.09  -9.23       +0.65 +0.78
    l,b,plx,pml,pmb 1000  -0.088 0.90 0.010   9.0                                    -0.27 -0.44 -0.20
            G,BP,RP 1000  -0.036 0.16 0.012   3.1 -168.75 -102.64 -0.08 -0.44  -9.49                  
        

## LaTeX

One `longtable` per file, wrapped in a group that sets `\tabcolsep`. The
first page carries the full or short caption and the `\label`; pages the
table continues onto repeat the header under the `cont` caption.

In [6]:
def _longtable(align, header, body, cap_first, cap_cont, label, ncol):
    out = []
    if TABCOLSEP:
        out.append(r'\begingroup\setlength{\tabcolsep}{' + TABCOLSEP + '}')
    out += [r'\begin{longtable}{' + align + '}',
            rf'\caption{{{cap_first}}}\label{{{label}}}\\',
            r'\toprule', header, r'\midrule', r'\endfirsthead',
            rf'\caption[]{{{cap_cont}}}\\',
            r'\toprule', header, r'\midrule', r'\endhead',
            r'\midrule', rf'\multicolumn{{{ncol}}}{{r}}{{continued}}\\',
            r'\endfoot', r'\bottomrule', r'\endlastfoot',
            *body, r'\end{longtable}']
    if TABCOLSEP:
        out.append(r'\endgroup')
    return out


def _n_info(t):
    """(show the n column?, the <<N>> text for the caption)."""
    ns = sorted(set(int(v) for v in t['n']))
    show = SHOW_N if SHOW_N in (True, False) else len(ns) > 1
    text = str(ns[0]) if len(ns) == 1 else f'up to {ns[-1]}'
    return show, text


def _pattern_cell(cond):
    return short_pattern(cond, FEATURES, symbols=True)


def tarp_lines(t, show_n):
    head = ['Cond.\\ Pattern'] + (['$n$'] if show_n else []) + [
        r'$\max |\Delta|$', '$c$', r'$\sigma_{\rm TARP}$',
        r'$\frac{|\Delta|}{\sigma_{\rm TARP}}$']
    body = []
    for r in t.itertuples():
        cells = [_pattern_cell(r.cond)] + ([str(r.n)] if show_n else [])
        cells += [_fmt_signed(r.max_dev, DEV_DECIMALS),
                  _fmt(r.cred_at_max, CRED_DECIMALS),
                  _fmt(r.sigma, SIGMA_DECIMALS),
                  _fmt(r.ratio, RATIO_DECIMALS)]
        body.append(' & '.join(cells) + r' \\')
    align = 'l ' + ('r ' if show_n else '') + 'rrrr'
    return align, ' & '.join(head) + r' \\', body, len(head)


def skill_lines(t, wide, cls, show_n):
    feats = CLASSES[cls]
    head = ['Cond.\\ Pattern'] + (['$n$'] if show_n else []) + [math(f) for f in feats]
    body = []
    for r in t.itertuples():
        cells = [_pattern_cell(r.cond)] + ([str(r.n)] if show_n else [])
        cells += [_fmt_signed(wide.at[r.p_idx, f], SKILL_DECIMALS) for f in feats]
        body.append(' & '.join(cells) + r' \\')
    align = 'l ' + ('r ' if show_n else '') + 'r' * len(feats)
    return align, ' & '.join(head) + r' \\', body, len(head)


def write_tables(datasets=None, classes=None, kinds=None, out_dir=None):
    """Write one .tex per (kind, dataset, class); returns the paths in order."""
    datasets = datasets or DATASETS
    classes = classes or CLASSES_TO_WRITE or list(CLASSES)
    kinds = kinds or KINDS
    out_dir = Path(out_dir or TABLE_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    templates = {'tarp': TARP_CAPTION, 'skill': SKILL_CAPTION}

    written = []
    for kind in kinds:
        for ds in datasets:
            first_label = None
            for cls in classes:
                t, wide = table_frames(ds, cls)
                if t.empty:
                    print(f'  skip {kind} {ds} ({class_letter(cls)}): no usable curves')
                    continue
                show_n, n_text = _n_info(t)
                letter = class_letter(cls)
                label = LABEL_FMT.format(kind=kind, dataset=ds, letter=letter)
                tokens = dict(N=n_text, DATASET=DS_CAPTION.get(ds, ds),
                              CLASS=letter, FEATURES=math(CLASSES[cls]),
                              FIRST_TABLE=first_label or label,
                              EQ_DEV=EQ_DEV_LABEL, EQ_ERR=EQ_ERR_LABEL,
                              RMSE_POOL=RMSE_POOL_SENTENCE[RMSE_SOURCE])
                cap = fill(templates[kind]['short' if first_label else 'full'], **tokens)
                cap_cont = fill(templates[kind]['cont'], **tokens)
                first_label = first_label or label

                if kind == 'tarp':
                    align, header, body, ncol = tarp_lines(t, show_n)
                else:
                    align, header, body, ncol = skill_lines(t, wide, cls, show_n)

                src = META[ds]['path'].name
                lines = ([f'% Generated by CFM_TARP_Skill_Tables.ipynb from {src}',
                          f'% {kind} table, dataset {ds}, class {cls}, '
                          f'RMSE_SOURCE={RMSE_SOURCE}',
                          '% Requires \\usepackage{booktabs, longtable}']
                         + _longtable(align, header, body, cap, cap_cont, label, ncol)
                         + [''])
                p = out_dir / f'{kind}_{ds}_{letter}.tex'
                p.write_text('\n'.join(lines))
                written.append(p)
                print(f'  wrote {p}   ({len(body)} rows)')
    return written


def write_preview(paths, out_dir=None):
    """A standalone document that \\input's every table, for a layout check.

    It defines the two equation labels so the captions' references resolve."""
    out_dir = Path(out_dir or TABLE_DIR)
    doc = [r'\documentclass[10pt]{article}',
           r'\usepackage[margin=2cm]{geometry}',
           r'\usepackage{booktabs}', r'\usepackage{longtable}',
           r'\usepackage{amsmath}', r'\usepackage{hyperref}',
           r'\begin{document}',
           "% Stand-ins for the paper's Equations A1 and A2.",
           r'\begin{equation}\sigma_{\rm TARP} = \sqrt{\frac{c(1-c)}{n}}'
           r'\label{' + EQ_ERR_LABEL + r'}\end{equation}',
           r'\begin{equation}|\Delta| \equiv |{\rm ECP} - c|'
           r'\label{' + EQ_DEV_LABEL + r'}\end{equation}', '']
    for p in paths:
        doc += [r'\input{' + Path(p).stem + '}', '']
    doc += [r'\end{document}', '']
    f = out_dir / 'tables_preview.tex'
    f.write_text('\n'.join(doc))
    print(f'  wrote {f}  (compile with pdflatex, twice, inside {out_dir})')
    return f

In [7]:
TABLE_FILES = write_tables()

if WRITE_PREVIEW:
    write_preview(TABLE_FILES)

if SAVE_CSV:
    TARP_DF.drop(columns='cond').assign(
        cond=[','.join(c) for c in TARP_DF['cond']]).to_csv(
        Path(TABLE_DIR) / 'tarp_numbers.csv', index=False)
    SKILL_DF.drop(columns='cond').assign(
        cond=[','.join(c) for c in SKILL_DF['cond']]).to_csv(
        Path(TABLE_DIR) / 'skill_numbers.csv', index=False)
    print(f'  wrote {Path(TABLE_DIR) / "tarp_numbers.csv"} and skill_numbers.csv')

  skip tarp omega_cen_not_A (a): no usable curves
  skip tarp omega_cen_not_A (b): no usable curves
  wrote tables_tarp_skill/tarp_omega_cen_not_A_c.tex   (29 rows)
  wrote tables_tarp_skill/tarp_omega_cen_not_A_d.tex   (47 rows)
  skip skill omega_cen_not_A (a): no usable curves
  skip skill omega_cen_not_A (b): no usable curves
  wrote tables_tarp_skill/skill_omega_cen_not_A_c.tex   (29 rows)
  wrote tables_tarp_skill/skill_omega_cen_not_A_d.tex   (47 rows)
  wrote tables_tarp_skill/tables_preview.tex  (compile with pdflatex, twice, inside tables_tarp_skill)
  wrote tables_tarp_skill/tarp_numbers.csv and skill_numbers.csv
